# 실험: Colab T4에서 torch 2.4.0+cu124 정확히 맞추기

기본 설치는 `torch==2.4.0` → PyPI 기본 빌드(cu121)가 깔린다. Pod 이미지는 **cu124**이므로
PyTorch **cu124 인덱스**에서 받아 정확히 일치시키고, T4 GPU에서 실제 CUDA 연산까지 되는지 확인한다.

## 1. uv + Python 3.11.9 venv

In [ ]:
!pip install -q uv
!uv venv --python 3.11.9 /content/py311
!/content/py311/bin/python --version
!nvidia-smi --query-gpu=name,driver_version --format=csv,noheader

## 2. torch 계열을 cu124 인덱스에서 설치

In [ ]:
!uv pip install --python /content/py311/bin/python \
    torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 \
    --index-url https://download.pytorch.org/whl/cu124
print('설치 완료')

## 3. 검증 — 버전이 +cu124 인지 + 실제 GPU 연산

In [ ]:
%%writefile cu124_check.py
import torch
print('torch        ', torch.__version__)
print('built cuda    ', torch.version.cuda)
print('cudnn         ', torch.backends.cudnn.version())
print('cuda available', torch.cuda.is_available())
assert torch.__version__ == '2.4.0+cu124', 'cu124 빌드가 아님: ' + torch.__version__
if torch.cuda.is_available():
    print('device        ', torch.cuda.get_device_name(0))
    x = torch.randn(2000, 2000, device='cuda')
    y = (x @ x).sum().item()
    torch.cuda.synchronize()
    print('CUDA matmul OK, sum =', round(y, 2))
    print('RESULT: cu124 + T4 GPU 연산 정상')
else:
    print('RESULT: cu124는 설치됐으나 GPU 미인식 (CPU 런타임?)')

In [ ]:
!/content/py311/bin/python cu124_check.py